In [4]:
# !pip install vl-convert-python -q
# !pip install "vegafusion[embed]" -q

In [1]:
import numpy as np
import pandas as pd
import altair as alt
import ipywidgets as widgets



alt.data_transformers.enable("vegafusion")

from IPython.display import display, clear_output


# -----------------------------------
# Lyapunov exponent
# -----------------------------------

def lyapunov_exponent(r, x0=0.1, burn_in=500, n=2000):

    x = x0

    # Let transient behavior disappear
    for _ in range(burn_in):
        x = r * x * (1 - x)

    logs = []

    for _ in range(n):

        x = r * x * (1 - x)

        derivative = abs(
            r * (1 - 2 * x)
        )

        if derivative > 0:
            logs.append(
                np.log(derivative)
            )

    return np.mean(logs)


# -----------------------------------
# Long-run states for bifurcation
# -----------------------------------

def bifurcation_points(
    r,
    x0=0.1,
    burn_in=500,
    keep=100
):

    x = x0

    # Remove transient
    for _ in range(burn_in):
        x = r * x * (1 - x)

    values = []

    # Keep long-run behavior
    for _ in range(keep):

        x = r * x * (1 - x)

        values.append(x)

    return values


# -----------------------------------
# Slider
# -----------------------------------

r_max_slider = widgets.FloatSlider(
    value=2.50,
    min=2.50,
    max=4.00,
    step=0.01,
    description="r max:",
    continuous_update=False,
    readout_format=".2f",
    layout=widgets.Layout(width="500px")
)

plot_output = widgets.Output()


# -----------------------------------
# Draw both diagrams
# -----------------------------------

def draw_complexity(*args):

    r_max = r_max_slider.value


    # ===================================
    # r values revealed so far
    # ===================================

    r_values = np.arange(
        2.50,
        r_max + 0.001,
        0.01
    )


    # ===================================
    # 1. LYAPUNOV
    # ===================================

    lambda_values = [
        lyapunov_exponent(r)
        for r in r_values
    ]

    df_lyapunov = pd.DataFrame({
        "r": r_values,
        "lambda": lambda_values
    })


    # Adaptive lower y-limit

    min_lambda = np.min(
        lambda_values
    )

    y_min = min(
        -1.5,
        min_lambda - 0.1
    )


    lyapunov_curve = (
        alt.Chart(df_lyapunov)
        .mark_line(
            point=True,
            strokeWidth=1.5
        )
        .encode(

            x=alt.X(
                "r:Q",
                title=None,
                scale=alt.Scale(
                    domain=[2.5, 4.0]
                )
            ),

            y=alt.Y(
                "lambda:Q",
                title="Lyapunov exponent (λ)",
                scale=alt.Scale(
                    domain=[y_min, 0.8]
                )
            ),

            tooltip=[
                alt.Tooltip(
                    "r:Q",
                    title="r",
                    format=".2f"
                ),

                alt.Tooltip(
                    "lambda:Q",
                    title="λ",
                    format=".4f"
                )
            ]
        )
    )


    # λ = 0

    zero_line = (
        alt.Chart(
            pd.DataFrame({
                "lambda": [0]
            })
        )
        .mark_rule(
            strokeDash=[6, 4],
            strokeWidth=2
        )
        .encode(
            y="lambda:Q"
        )
    )


    # Current frontier

    current = pd.DataFrame({
        "r": [r_values[-1]],
        "lambda": [lambda_values[-1]]
    })

    current_point = (
        alt.Chart(current)
        .mark_point(
            filled=True,
            size=150
        )
        .encode(
            x="r:Q",
            y="lambda:Q"
        )
    )


    lyapunov_chart = (
        (
            lyapunov_curve
            + zero_line
            + current_point
        )
        .properties(
            width=800,
            height=300,
            title=(
                f"Exploring complexity — "
                f"r = 2.50 → {r_max:.2f}"
            )
        )
    )


    # ===================================
    # 2. BIFURCATION
    # ===================================

    bifurcation_data = []

    for r in r_values:

        long_run_values = bifurcation_points(r)

        for x in long_run_values:

            bifurcation_data.append({
                "r": r,
                "x": x
            })


    df_bifurcation = pd.DataFrame(
        bifurcation_data
    )


    bifurcation_chart = (
        alt.Chart(df_bifurcation)
        .mark_point(
            size=5,
            opacity=0.55
        )
        .encode(

            x=alt.X(
                "r:Q",
                title="r",
                scale=alt.Scale(
                    domain=[2.5, 4.0]
                )
            ),

            y=alt.Y(
                "x:Q",
                title="Long-run state",
                scale=alt.Scale(
                    domain=[0, 1]
                )
            ),

            tooltip=[
                alt.Tooltip(
                    "r:Q",
                    title="r",
                    format=".2f"
                ),

                alt.Tooltip(
                    "x:Q",
                    title="x",
                    format=".4f"
                )
            ]
        )
        .properties(
            width=800,
            height=300,
            title="Bifurcation diagram"
        )
    )


    # ===================================
    # Combine
    # ===================================

    chart = (
        alt.vconcat(
            lyapunov_chart,
            bifurcation_chart,
            spacing=15
        )
        .resolve_scale(
            x="shared"
        )
    )


    with plot_output:

        clear_output(wait=True)
        display(chart)


# -----------------------------------
# Connect slider
# -----------------------------------

r_max_slider.observe(
    draw_complexity,
    names="value"
)


display(
    r_max_slider,
    plot_output
)

draw_complexity()

FloatSlider(value=2.5, continuous_update=False, description='r max:', layout=Layout(width='500px'), max=4.0, m…

Output()